# Execution — Working the Book Like a Trading Desk

Takes Signal's weekly picks (`signal_picks.csv`) and simulates actually filling
them, the way an equities execution desk would: benchmarked against **arrival
price** and **VWAP**, the two standard metrics any execution trader is judged on.

**Terms used below, if you haven't seen them:**
- **Arrival price** — the price at the moment the order was received (here: the
  week's opening price). The baseline every fill is measured against.
- **VWAP** — volume-weighted average price over the execution horizon (here: the
  week). "Did we beat VWAP?" is the single most common execution report line
  on a real desk.
- **Implementation shortfall** — how much worse (or better) your actual average
  fill was vs. arrival price, in basis points (bps). This is the real cost of
  trading, separate from whether the stock pick itself was good.
- **Market impact** — the price moves against you simply because you're
  trading; bigger orders relative to average volume move the price more.

**Two execution strategies compared, per order:**
- **All-at-once** — the full order hits the market immediately at the week's
  open. Minimizes timing risk, maximizes market impact (one big print).
- **TWAP** — the order is split evenly across the week's trading days instead
  of firing all at once. Lower impact per slice, but now you're exposed to
  the stock drifting against you over the week (timing risk). This is the
  textbook execution-desk tradeoff.

**Data constraint (read this before judging the numbers):** we only have daily
OHLCV bars, not intraday ticks, so "TWAP" here means splitting the order across
the week's *daily* opens rather than true intraday slicing. This is a real
simplification a professional desk wouldn't accept, but it's what the data
allows and it still captures the core impact-vs-timing-risk tradeoff.

**Order sizing:** each leg is sized as a **% of that ticker's `avg_volume_20d`**
(not a flat dollar amount) — sizing by % of ADV is how real desks actually size
orders, and it makes market impact comparable across a $50 stock and a $1,600
stock. Configurable via `PCT_OF_ADV`.

Output: `execution_fills.csv`, one row per `(week_number, ticker)` with fills
and cost metrics for both strategies.

In [1]:
# ============================================================
# CONFIG
# ============================================================
import pandas as pd
import numpy as np

DATA_CSV = "semiconductor_data.csv"
SIGNAL_CSV = "signal_picks.csv"
OUTPUT_CSV = "execution_fills.csv"

PCT_OF_ADV = 0.05        # each order = 5% of the ticker's avg_volume_20d
VOL_LOOKBACK = 20        # trading days used to estimate daily volatility (sigma)
IMPACT_COEF = 1.0        # calibration constant (Y) in the square-root impact model:
                          #   impact_bps = IMPACT_COEF * sigma_daily * sqrt(shares / ADV) * 10000
                          # This functional form (cost scales with the SQUARE ROOT of
                          # order size relative to ADV) is the industry-standard shape
                          # used in execution cost models (Almgren-Chriss and similar).
                          # IMPACT_COEF=1.0 is an illustrative default, not calibrated
                          # to real fill data -- flag this if the team wants to tune it.

print(f"Order size: {PCT_OF_ADV:.0%} of ADV | Impact coef: {IMPACT_COEF} | Vol lookback: {VOL_LOOKBACK}d")

Order size: 5% of ADV | Impact coef: 1.0 | Vol lookback: 20d


In [2]:
# ============================================================
# 1. LOAD DATA + SIGNAL, COMPUTE TRAILING VOLATILITY
# ============================================================
prices = pd.read_csv(DATA_CSV, parse_dates=["date"]).sort_values(["ticker", "date"]).reset_index(drop=True)
picks = pd.read_csv(SIGNAL_CSV, parse_dates=["signal_date"])

# Daily volatility per ticker, known as of the START of each day (shift(1) so
# a day's own return can't leak into the sigma used to size that same day's
# impact -- same lookahead discipline as the rest of the pipeline).
prices["daily_return"] = prices.groupby("ticker")["close"].pct_change()
prices["sigma_20d"] = (
    prices.groupby("ticker")["daily_return"]
    .transform(lambda s: s.rolling(VOL_LOOKBACK).std().shift(1))
)

print(f"Loaded {len(prices)} price rows, {len(picks)} signal picks.")
print(f"Sigma available for {prices['sigma_20d'].notna().sum()}/{len(prices)} rows.")

Loaded 10040 price rows, 1000 signal picks.
Sigma available for 9620/10040 rows.


In [3]:
# ============================================================
# 2. PER-WEEK BENCHMARKS: ARRIVAL PRICE + VWAP
# ============================================================
# Arrival price = the open on the week's first trading day (when the order
# "arrives" at the desk, per Signal's timing convention).
# VWAP = volume-weighted average of each day's typical price ((H+L+C)/3),
# the standard proxy for VWAP when only daily bars (not ticks) are available.
prices["typical_price"] = (prices["high"] + prices["low"] + prices["close"]) / 3

week_groups = prices.groupby(["ticker", "week_number"])

week_stats = week_groups.apply(
    lambda g: pd.Series({
        "arrival_price": g.sort_values("date")["open"].iloc[0],
        "arrival_adv": g.sort_values("date")["avg_volume_20d"].iloc[0],
        "arrival_sigma": g.sort_values("date")["sigma_20d"].iloc[0],
        "week_vwap": np.average(g["typical_price"], weights=g["volume"]),
        "n_days": len(g),
        "daily_opens": list(g.sort_values("date")["open"]),
    }),
    include_groups=False,
).reset_index()

print(f"Computed weekly benchmarks for {len(week_stats)} (ticker, week) combinations.")
print(f"Trading days per week -- min: {week_stats['n_days'].min()}, max: {week_stats['n_days'].max()}, "
      f"most common: {week_stats['n_days'].mode().iloc[0]}")

Computed weekly benchmarks for 2100 (ticker, week) combinations.
Trading days per week -- min: 3, max: 5, most common: 5


In [4]:
# ============================================================
# 3. SIZE EACH ORDER (% of ADV) AND ATTACH BENCHMARKS
# ============================================================
orders = picks.merge(week_stats, on=["week_number", "ticker"], how="left")

missing_stats = orders[orders["arrival_price"].isna()]
if len(missing_stats) > 0:
    print(f"WARNING: {len(missing_stats)} picks have no matching price data -- dropping them:")
    print(missing_stats[["week_number", "ticker", "side"]])
orders = orders.dropna(subset=["arrival_price", "arrival_adv", "arrival_sigma"]).reset_index(drop=True)

orders["shares"] = (orders["arrival_adv"] * PCT_OF_ADV).round().astype(int)
# side_mult: +1 for long (buying -- a higher fill is bad), -1 for short
# (selling -- a lower fill is bad). Used everywhere below to keep cost sign
# conventions consistent regardless of side.
orders["side_mult"] = np.where(orders["side"] == "long", 1, -1)

print(f"{len(orders)} orders sized. Example order sizes (shares) by ticker:")
print(orders.groupby("ticker")["shares"].first().sort_values().to_string())

1000 orders sized. Example order sizes (shares) by ticker:
ticker
MPWR       34510
SWKS      102697
ASML      123434
NXPI      125364
ADI       219485
ON        255567
TXN       256666
STM       260387
AMAT      298850
MCHP      311803
QCOM      380680
KLAC      495958
MRVL      504986
TSM       593730
LRCX      633485
AVGO     1431627
MU       1566271
AMD      1631080
INTC     5022912
NVDA    14545309


In [5]:
# ============================================================
# 4. SIMULATE FILLS -- ALL-AT-ONCE vs. TWAP
# ============================================================
# Square-root market impact: cost (as a fraction of price) = IMPACT_COEF *
# sigma_daily * sqrt(order_shares / ADV). Impact always pushes the fill price
# AWAY from arrival in the direction that hurts you (buying pushes price up,
# selling pushes it down) -- that's what side_mult encodes.
def impact_frac(shares, adv, sigma):
    return IMPACT_COEF * sigma * np.sqrt(shares / adv)

# --- All-at-once: full size executed immediately at the week's open ---
orders["impact_allatonce"] = impact_frac(orders["shares"], orders["arrival_adv"], orders["arrival_sigma"])
orders["fill_allatonce"] = orders["arrival_price"] * (1 + orders["side_mult"] * orders["impact_allatonce"])

# --- TWAP: split evenly across the week's daily opens ---
# Each child slice is smaller, so each has proportionally less impact (this is
# the whole point of TWAP -- sqrt() is concave, so N smaller slices cost less
# in total impact than one big one). But each slice fills at THAT day's actual
# open, so the strategy is now exposed to however the stock actually moved
# over the week -- that's the timing risk you're trading impact cost for.
def twap_avg_fill(row):
    n = row["n_days"]
    child_shares = row["shares"] / n
    child_impact = impact_frac(child_shares, row["arrival_adv"], row["arrival_sigma"])
    child_fills = [
        day_open * (1 + row["side_mult"] * child_impact)
        for day_open in row["daily_opens"]
    ]
    return np.mean(child_fills)

orders["fill_twap"] = orders.apply(twap_avg_fill, axis=1)

print("Fills computed. Sample:")
print(orders[["week_number", "ticker", "side", "shares", "arrival_price", "fill_allatonce", "fill_twap"]].head(10))

Fills computed. Sample:
   week_number ticker   side    shares  arrival_price  fill_allatonce  \
0            6   AVGO   long   1431627     172.157265      173.147459   
1            6    AMD   long   1631080     171.080002      171.989947   
2            6   NVDA   long  14545309     124.795000      125.569828   
3            6   INTC   long   5022912      22.400000       22.523959   
4            6     MU   long   1566271     101.635822      102.587180   
5            6   SWKS  short    102697      89.401783       88.984092   
6            6    STM  short    260387      27.422301       27.273038   
7            6    TXN  short    256666     189.818559      189.116108   
8            6   NXPI  short    125364     229.580613      228.568337   
9            6   MCHP  short    311803      72.731959       72.389752   

    fill_twap  
0  176.565099  
1  170.570766  
2  131.214881  
3   22.884496  
4  102.500864  
5   88.803622  
6   27.210956  
7  190.165718  
8  226.610208  
9   72.52413

In [6]:
# ============================================================
# 5. COST METRICS (bps) -- how a real execution desk reports quality
# ============================================================
def bps(fill, benchmark, side_mult):
    return side_mult * (fill - benchmark) / benchmark * 10000

orders["shortfall_allatonce_bps"] = bps(orders["fill_allatonce"], orders["arrival_price"], orders["side_mult"])
orders["shortfall_twap_bps"] = bps(orders["fill_twap"], orders["arrival_price"], orders["side_mult"])
orders["vs_vwap_allatonce_bps"] = bps(orders["fill_allatonce"], orders["week_vwap"], orders["side_mult"])
orders["vs_vwap_twap_bps"] = bps(orders["fill_twap"], orders["week_vwap"], orders["side_mult"])

# ============================================================
# 6. FINALIZE + SAVE
# ============================================================
final_cols = [
    "week_number", "ticker", "side", "rank", "shares",
    "arrival_price", "week_vwap",
    "fill_allatonce", "shortfall_allatonce_bps", "vs_vwap_allatonce_bps",
    "fill_twap", "shortfall_twap_bps", "vs_vwap_twap_bps",
]
execution_df = orders[final_cols].sort_values(["week_number", "side", "rank"]).reset_index(drop=True)
execution_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(execution_df)} rows to {OUTPUT_CSV}")

Saved 1000 rows to execution_fills.csv


In [7]:
# ============================================================
# 7. DESK REPORT -- how would this actually get reported to a PM?
# ============================================================
print("=" * 60)
print("SAMPLE OUTPUT")
print("=" * 60)
print(execution_df.head(10))

print("\n" + "=" * 60)
print("SANITY CHECKS")
print("=" * 60)

nan_counts = execution_df.isna().sum()
nan_counts = nan_counts[nan_counts > 0]
print(f"[{'OK' if len(nan_counts) == 0 else 'FAIL'}] NaNs in output: "
      f"{'none' if len(nan_counts) == 0 else dict(nan_counts)}")

neg_shares = (execution_df["shares"] <= 0).sum()
print(f"[{'OK' if neg_shares == 0 else 'FAIL'}] Orders with non-positive share count: {neg_shares}")

print("\n" + "=" * 60)
print("EXECUTION QUALITY: ALL-AT-ONCE vs. TWAP")
print("=" * 60)
summary = execution_df[["shortfall_allatonce_bps", "shortfall_twap_bps",
                         "vs_vwap_allatonce_bps", "vs_vwap_twap_bps"]].agg(["mean", "std"])
print(summary.round(2))

twap_wins = (execution_df["shortfall_twap_bps"] < execution_df["shortfall_allatonce_bps"]).mean()
print(f"\n[INFO] TWAP beat all-at-once on implementation shortfall in {twap_wins:.0%} of orders.")

print(f"[INFO] Avg all-at-once shortfall: {execution_df['shortfall_allatonce_bps'].mean():.1f} bps "
      f"(this is pure market impact, since all-at-once has zero timing risk by construction).")
print(f"[INFO] Avg TWAP shortfall: {execution_df['shortfall_twap_bps'].mean():.1f} bps "
      f"(lower impact per slice, but now includes the week's actual price drift).")
print("If TWAP's average shortfall is higher despite lower per-slice impact, that means timing risk "
      "(the stock moving against you while you wait to finish the order) is outweighing the impact "
      "savings -- a real, common outcome for momentum-driven names that keep trending during the week.")

SAMPLE OUTPUT
   week_number ticker   side  rank    shares  arrival_price   week_vwap  \
0            6   AVGO   long     1   1431627     172.157265  178.032180   
1            6    AMD   long     2   1631080     171.080002  168.724571   
2            6   NVDA   long     3  14545309     124.795000  131.469670   
3            6   INTC   long     4   5022912      22.400000   23.022242   
4            6     MU   long     5   1566271     101.635822  103.009644   
5            6   SWKS  short     1    102697      89.401783   89.640348   
6            6    STM  short     2    260387      27.422301   27.393874   
7            6    TXN  short     3    256666     189.818559  191.566374   
8            6   NXPI  short     4    125364     229.580613  228.148564   
9            6   MCHP  short     5    311803      72.731959   73.051907   

   fill_allatonce  shortfall_allatonce_bps  vs_vwap_allatonce_bps   fill_twap  \
0      173.147459                57.516831            -274.372891  176.565099  